# Edge-IIoTset Open-Set Federated IDS — Reproducible Dataset Migration Benchmark

**Question:** Does the dataset change improve leave-one-attack-type-out detection, and does a lightweight RBF-KAN or post-hoc multi-prototype scoring outperform an MLP at validation-calibrated false-positive rates?

**No claims of new algorithm superiority.** This is a benchmark and feasibility diagnostic. Five **simulated** non-IID clients; three seeds; up to four held-out attack types. Every held-out attack type is excluded from training, validation and prototype estimation. Preprocessing parameters are fitted on training rows only. Duplicates with identical feature vectors are eliminated before partitioning. Scores are calibrated using validation benign records only. We report actual test FPR separately. Test uses a capped subset: precision/F1/AUPRC are not production-prevalence metrics.

Dataset: original author Edge-IIoTset release (`DNN-EdgeIIoT-dataset.csv`, alternatively `ML-EdgeIIoT-dataset.csv`). For reproducible unattended runs, **attach** `mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot` to the Kaggle notebook or enable Internet for KaggleHub.

Methods: `FedAvgMLP`, `FedAvgKAN`, `LocalKAN`, `FedAvgKAN_SingleProto`, `FedAvgKAN_MultiProto`. Prototype alternatives are **post-hoc decision functions** sharing the FedAvgKAN encoder; they are not separately trained novel FL algorithms. The KAN is a lightweight Gaussian RBF expansion, not the original B-spline KAN.

**Compute:** two Kaggle T4 GPUs are used in separate worker processes when available. The notebook saves a final `EdgeIIoT_OpenSet_Results.zip` under `/kaggle/working`.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'kagglehub'])
from pathlib import Path
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data/edge_benchmark_notebook_test')
work.mkdir(parents=True, exist_ok=True)
print('Work directory:',work)

In [ ]:
from pathlib import Path
engine_source = '"""Edge-IIoTset open-set FL benchmark. All in-notebook/source content is English.\nResearch diagnostic only; simulated clients and test subsampling are explicitly documented.\nRequires the official ML/DNN Edge-IIoTset CSV attached to Kaggle or KaggleHub access.\n"""\nimport os, sys, json, time, random, hashlib, traceback, zipfile\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, confusion_matrix\nfrom sklearn.cluster import MiniBatchKMeans\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nROOT=Path(os.getenv(\'EDGE_BENCH_ROOT\',\'/kaggle/working\' if Path(\'/kaggle/working\').exists() else \'/mnt/data/edge_benchmark_local\'))\nOUT=ROOT/\'EdgeIIoT_OpenSet_Results\'; OUT.mkdir(parents=True,exist_ok=True)\nCFG=dict(seeds=[0,42,3407],families=[\'Port_Scanning\',\'DDoS_UDP\',\'SQL_injection\',\'Ransomware\'],n_clients=5,\n rounds=7,local_steps=12,batch_size=128,lr=0.0015,hidden=24,embedding=12,rbf_bases=5,\n per_class_cap=18000,benign_cap=50000,unknown_test_cap=16000,valid_frac=0.18,test_benign_frac=0.23,\n drift_alpha=0.4,benign_centers=4,attack_centers=3,score_weight=0.45,\n thresholds=[0.01,0.05],max_raw_rows=2400000,chunk_size=70000,comparison_methods=[\'FedAvgMLP\',\'FedAvgKAN\',\'LocalKAN\',\'FedAvgKAN_SingleProto\',\'FedAvgKAN_MultiProto\'],\n archive=\'EdgeIIoT_OpenSet_Results.zip\')\nif os.getenv(\'EDGE_BENCH_SEEDS\'):\n CFG[\'seeds\']=[int(t) for t in os.environ[\'EDGE_BENCH_SEEDS\'].split(\',\') if t.strip()]\nif os.getenv(\'EDGE_BENCH_SMOKE\')==\'1\':\n CFG.update(seeds=[0],families=[\'Port_Scanning\',\'DDoS_UDP\'],n_clients=3,rounds=2,local_steps=2,per_class_cap=500,benign_cap=700,unknown_test_cap=400,hidden=12,embedding=8)\n\ndef seed_all(s):\n random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.set_num_threads(min(2,os.cpu_count() or 2))\n if torch.cuda.is_available():torch.cuda.manual_seed_all(s)\n\ndef jsave(path,obj):\n Path(path).write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),encoding=\'utf8\')\n\ndef norm(s):return \'\'.join(c for c in str(s).strip().lower() if c.isalnum())\n\ndef get_dataset_file():\n if os.getenv(\'EDGE_BENCH_SMOKE\')==\'1\':\n  p=ROOT/\'synthetic_smoke_only.csv\'\n  if not p.exists():\n   rng=np.random.default_rng(9);a=[]\n   for family,n,shift in [(\'Normal\',1400,0),(\'Port_Scanning\',450,1.3),(\'DDoS_UDP\',470,1.8),(\'SQL_injection\',300,2.2),(\'Ransomware\',290,2.5),(\'MITM\',360,1.4)]:\n    x=rng.normal(size=(n,15));x[:,0]+=shift;x[:,1]+=shift*.5\n    q=pd.DataFrame(x,columns=[\'tcp.flags\',\'tcp.len\',\'udp.time_delta\',\'icmp.seq_le\',\'mqtt.len\',\'arp.opcode\',\'tcp.connection.syn\',\'tcp.connection.rst\',\'dns.qry.type\',\'dns.qry.qu\',\'mqtt.msgtype\',\'mbtcp.len\',\'http.content_length\',\'tcp.flags.ack\',\'tcp.connection.fin\'])\n    q[\'Attack_type\']=family;q[\'Attack_label\']=0 if family==\'Normal\' else 1;a.append(q)\n   pd.concat(a,ignore_index=True).to_csv(p,index=False)\n  return p,\'SYNTHETIC_SMOKE_ONLY\'\n roots=[]\n if os.environ.get(\'EDGE_IIOT_CSV\'):roots.append(Path(os.environ[\'EDGE_IIOT_CSV\']))\n roots.extend([Path(\'/kaggle/input\'),ROOT/\'dataset\'])\n candidates=[]\n for r in roots:\n  if r.is_file() and r.suffix.lower()==\'.csv\':candidates.append(r)\n  if r.is_dir():candidates.extend(r.rglob(\'*.csv\'))\n for p in sorted(candidates,key=lambda x:(0 if \'dnn-edgeiiot\' in x.name.lower() else 1 if \'ml-edgeiiot\' in x.name.lower() else 2,len(str(x)))):\n  try:\n   h=pd.read_csv(p,nrows=0).columns\n   if {\'attacktype\',\'attacklabel\'}.issubset({norm(c) for c in h}):return p,\'attached:\'+str(p)\n  except Exception:pass\n # KaggleHub downloads the published author Kaggle dataset, using Kaggle environment credentials.\n try:\n  import kagglehub\n  p=Path(kagglehub.dataset_download(\'mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot\'))\n  for f in sorted(p.rglob(\'*.csv\'),key=lambda x:(0 if \'dnn-edgeiiot\' in x.name.lower() else 1,len(str(x)))):\n   try:\n    if {\'attacktype\',\'attacklabel\'}.issubset({norm(c) for c in pd.read_csv(f,nrows=0).columns}):return f,\'author_kagglehub:\'+str(f)\n   except Exception:pass\n except Exception as e:print(\'KaggleHub download unavailable:\',repr(e),flush=True)\n raise FileNotFoundError(\'Edge-IIoTset CSV not found. Attach the author Edge-IIoTset Kaggle dataset (ML-EdgeIIoT-dataset.csv or DNN-EdgeIIoT-dataset.csv) and enable Internet if downloading.\')\n\n# A conservative, prespecified packet/flow feature allowlist; excludes addresses, payloads,\n# checksums, unique stream identifiers and attack-specific HTTP strings to reduce shortcuts.\nFEATURES=[\'arp.opcode\',\'arp.hw.size\',\'icmp.seq_le\',\'icmp.unused\',\'http.content_length\',\'http.tls_port\',\n \'tcp.ack\',\'tcp.connection.fin\',\'tcp.connection.rst\',\'tcp.connection.syn\',\'tcp.connection.synack\',\n \'tcp.flags\',\'tcp.flags.ack\',\'tcp.len\',\'udp.time_delta\',\'dns.qry.qu\',\'dns.qry.type\',\n \'dns.retransmission\',\'dns.retransmit_request\',\'dns.retransmit_request_in\',\n \'mqtt.conflag.cleansess\',\'mqtt.conflags\',\'mqtt.hdrflags\',\'mqtt.len\',\n \'mqtt.msgtype\',\'mqtt.proto_len\',\'mqtt.topic_len\',\'mqtt.ver\',\'mbtcp.len\',\'mbtcp.unit_id\']\n\ndef read_dataset(path):\n head=pd.read_csv(path,nrows=0).columns.tolist();lab=next(c for c in head if norm(c)==\'attacktype\');bl=next(c for c in head if norm(c)==\'attacklabel\')\n choose=[c for c in head if norm(c) in {norm(x) for x in FEATURES} and c not in [lab,bl]]\n if len(choose)<8:raise ValueError(f\'Only {len(choose)} safe numeric features found in {path.name}; refusing unsupported schema.\')\n use=choose+[lab,bl];sample=[];n_all=0\n for chunk in pd.read_csv(path,usecols=use,chunksize=CFG[\'chunk_size\'],low_memory=False):\n  chunk=chunk.rename(columns={lab:\'family\',bl:\'binary_label\'})\n  chunk[\'family\']=chunk[\'family\'].astype(str).str.strip().str.replace(\' \',\'_\',regex=False)\n  chunk[\'binary_label\']=pd.to_numeric(chunk[\'binary_label\'],errors=\'coerce\')\n  chunk=chunk[chunk[\'binary_label\'].isin([0,1])];\n  for c in choose:chunk[c]=pd.to_numeric(chunk[c],errors=\'coerce\')\n  chunk=chunk.dropna(subset=[\'family\']);sample.append(chunk);n_all+=len(chunk)\n  if n_all>=CFG[\'max_raw_rows\']:break\n df=pd.concat(sample,ignore_index=True)\n pre_dedup=len(df)\n df=df.drop_duplicates(subset=choose,keep=\'first\').reset_index(drop=True)\n n_dedup=pre_dedup-len(df)\n # deterministic per-class sampling, independent of the random experiment seed\n rng=np.random.default_rng(1717);indices=[]\n for name,g in df.groupby(\'family\',sort=True):\n  cap=CFG[\'benign_cap\'] if int(g[\'binary_label\'].iloc[0])==0 else CFG[\'per_class_cap\']\n  ix=g.index.to_numpy();indices.extend(rng.choice(ix,size=min(len(ix),cap),replace=False).tolist())\n df=df.loc[sorted(indices)].reset_index(drop=True)\n uniq=df.groupby(\'family\')[\'binary_label\'].nunique()\n if (uniq>1).any():raise ValueError(\'Inconsistent attack labels within a family.\')\n return df,choose,dict(source_rows_scanned=n_all,exact_duplicate_feature_rows_removed=n_dedup,rows_after_capping=len(df),family_counts=df[\'family\'].value_counts().to_dict(),features=choose,schema=\'prespecified_numeric_safe_subset\',note=\'Class capping changes prevalence; test metrics use separate benign and unknown holdouts, not production prevalence.\')\n\ndef create_fold(df,features,held,seed):\n rng=np.random.default_rng(seed);fam=df[\'family\'].to_numpy();yb=df[\'binary_label\'].astype(int).to_numpy()\n unknown=np.where((fam==held)&(yb==1))[0];benign=np.where(yb==0)[0];known=np.where((yb==1)&(fam!=held))[0]\n if len(unknown)<20 or len(known)<70 or len(benign)<300:raise ValueError(\'Insufficient data for held-out family \'+held)\n # Unknown test examples are disjoint from training and validation by construction.\n unknown=rng.choice(unknown,min(len(unknown),CFG[\'unknown_test_cap\']),replace=False)\n benign_trainval,benign_test=train_test_split(benign,test_size=CFG[\'test_benign_frac\'],random_state=seed)\n benign_train,benign_val=train_test_split(benign_trainval,test_size=CFG[\'valid_frac\'],random_state=seed+1)\n known_train,known_val=train_test_split(known,test_size=CFG[\'valid_frac\'],random_state=seed+2,stratify=fam[known] if min(Counter(fam[known]).values())>=3 else None)\n train=np.r_[benign_train,known_train];val=np.r_[benign_val,known_val];test=np.r_[benign_test,unknown]\n for a,b in [(train,val),(train,test),(val,test)]:assert not set(a).intersection(set(b))\n assert all(fam[x]!=held for x in train) and all(fam[x]!=held for x in val)\n # fit data-dependent transforms ONLY on training rows\n X=df[features].to_numpy(np.float32,copy=True)\n X[~np.isfinite(X)]=np.nan\n imp=SimpleImputer(strategy=\'median\',keep_empty_features=True);Xtr=imp.fit_transform(X[train]);X=imp.transform(X)\n scaler=RobustScaler(quantile_range=(10,90));scaler.fit(Xtr);X=np.clip(scaler.transform(X),-12,12).astype(\'float32\')\n # Train-only feature scaling; no test distribution statistics enter fit.\n # Use a stratified Dirichlet allocation by known family to simulate heterogeneous clients.\n buckets=[[] for _ in range(CFG[\'n_clients\'])]\n for f in sorted(set(fam[train])):\n  ix=train[fam[train]==f].copy();rng.shuffle(ix)\n  prop=rng.dirichlet(np.full(CFG[\'n_clients\'],CFG[\'drift_alpha\']))\n  split=np.split(ix,np.cumsum(rng.multinomial(len(ix),prop))[:-1]);\n  for j,segment in enumerate(split):buckets[j].extend(segment.tolist())\n # Ensure every simulated client can train on enough rows.\n if min(map(len,buckets))<50:\n  shuffled=train.copy();rng.shuffle(shuffled);buckets=[x.tolist() for x in np.array_split(shuffled,CFG[\'n_clients\'])]\n return X,yb,fam,train,val,test,[np.array(b,dtype=int) for b in buckets],dict(heldout=held,seed=seed,n_train=len(train),n_val=len(val),n_test=len(test),n_unknown=len(unknown),n_benign_test=len(benign_test),train_family_counts=Counter(fam[train]),val_family_counts=Counter(fam[val]),test_family_counts=Counter(fam[test]),client_counts=[len(x) for x in buckets])\n\nclass RBFLayer(nn.Module):\n def __init__(self,ni,no,bases):\n  super().__init__();self.register_buffer(\'grid\',torch.linspace(-2.5,2.5,bases));self.scale=nn.Parameter(torch.ones(ni));self.lin=nn.Linear(ni*bases,no)\n def forward(self,x):\n  z=torch.exp(-((x.unsqueeze(-1)-self.grid)/(.8+torch.abs(self.scale).unsqueeze(-1)))**2)\n  return self.lin(z.flatten(1))\n\nclass Net(nn.Module):\n def __init__(self,d,kan):\n  super().__init__();self.kan=kan\n  if kan:self.a=RBFLayer(d,CFG[\'hidden\'],CFG[\'rbf_bases\']);self.b=RBFLayer(CFG[\'hidden\'],CFG[\'embedding\'],CFG[\'rbf_bases\'])\n  else:self.a=nn.Linear(d,CFG[\'hidden\']);self.b=nn.Linear(CFG[\'hidden\'],CFG[\'embedding\'])\n  self.head=nn.Linear(CFG[\'embedding\'],1)\n def embed(self,x):return torch.tanh(self.b(F.relu(self.a(x))))\n def forward(self,x):\n  z=self.embed(x);return self.head(z).flatten()\n\ndef local_fit(model,xx,yy,idx,device,seed):\n model=model.to(device).train();opt=torch.optim.AdamW(model.parameters(),lr=CFG[\'lr\'],weight_decay=1e-4)\n rng=np.random.default_rng(seed)\n # A shared fixed SGD-step count per client/method; balanced minibatches in training only.\n pos=idx[yy[idx]==1];neg=idx[yy[idx]==0]\n if not len(pos) or not len(neg):pool=idx\n for step in range(CFG[\'local_steps\']):\n  if len(pos) and len(neg):\n   a=rng.choice(pos,CFG[\'batch_size\']//2,replace=True);b=rng.choice(neg,CFG[\'batch_size\']-len(a),replace=True);take=np.r_[a,b];rng.shuffle(take)\n  else:take=rng.choice(pool,CFG[\'batch_size\'],replace=True)\n  x=torch.from_numpy(xx[take]).to(device);y=torch.from_numpy(yy[take].astype(\'float32\')).to(device)\n  opt.zero_grad(set_to_none=True);loss=F.binary_cross_entropy_with_logits(model(x),y);loss.backward();opt.step()\n return model.cpu()\n\ndef train_fed(X,y,clients,seed,kind,device):\n seed_all(seed);template=Net(X.shape[1],kan=(kind!=\'MLP\'));current={k:v.cpu().clone() for k,v in template.state_dict().items()}\n if kind==\'LOCAL\':\n  models=[]\n  for i,idx in enumerate(clients):\n   m=Net(X.shape[1],True);m.load_state_dict(current)\n   for r in range(CFG[\'rounds\']):m=local_fit(m,X,y,idx,device,seed+10000+i*100+r)\n   models.append(m)\n  return models,0,CFG[\'rounds\']*CFG[\'local_steps\']*len(clients)\n for r in range(CFG[\'rounds\']):\n  states=[]\n  for i,idx in enumerate(clients):\n   m=Net(X.shape[1],kind==\'KAN\');m.load_state_dict(current)\n   m=local_fit(m,X,y,idx,device,seed+10000+i*100+r)\n   states.append(m.state_dict())\n  counts=np.array([len(b) for b in clients],float);weights=counts/counts.sum()\n  current={name:sum(states[j][name].float()*float(weights[j]) for j in range(len(states))) for name in current}\n final=Net(X.shape[1],kind==\'KAN\');final.load_state_dict(current)\n comm=sum(v.numel()*v.element_size() for v in current.values())*len(clients)*CFG[\'rounds\']*2\n return final,comm,CFG[\'rounds\']*CFG[\'local_steps\']*len(clients)\n\n@torch.inference_mode()\ndef infer(model,X,idx,device):\n model.to(device).eval();scores=[];z=[]\n for sub in np.array_split(idx,max(1,int(np.ceil(len(idx)/2048)))):\n  if not len(sub):continue\n  inp=torch.from_numpy(X[sub]).to(device);scores.append(torch.sigmoid(model(inp)).cpu().numpy());z.append(model.embed(inp).cpu().numpy())\n model.cpu();return np.concatenate(scores),np.concatenate(z)\n\ndef fit_proto(z,y,train,clients,mode,seed):\n if mode==\'none\':return None\n # Federated prototype collection: each client computes local class centers,\n # server sees ONLY centers and counts, not individual training embeddings.\n prototypes=[]\n for label in [0,1]:\n  local_centers=[];local_weights=[]\n  for client_id,idx in enumerate(clients):\n   ids=idx[y[idx]==label]\n   if len(ids)==0:continue\n   if mode==\'single\':\n    centers=np.mean(z[ids],axis=0,keepdims=True)\n    counts=np.array([len(ids)])\n   else:\n    k=min(CFG[\'benign_centers\'] if label==0 else CFG[\'attack_centers\'],len(ids))\n    local=MiniBatchKMeans(n_clusters=k,random_state=seed+label+client_id*17,batch_size=512,n_init=3).fit(z[ids])\n    centers=local.cluster_centers_\n    counts=np.bincount(local.labels_,minlength=k)\n   for center,n in zip(centers,counts):\n    if n>0:local_centers.append(center);local_weights.append(int(n))\n  if not local_centers:raise ValueError(\'No class support among clients for prototype estimation.\')\n  local_centers=np.asarray(local_centers,dtype=np.float32)\n  if mode==\'single\':\n   centers=np.average(local_centers,axis=0,weights=local_weights)[None,:]\n  else:\n   k=min(CFG[\'benign_centers\'] if label==0 else CFG[\'attack_centers\'],len(local_centers))\n   server=MiniBatchKMeans(n_clusters=k,random_state=seed+label+177,batch_size=128,n_init=3)\n   server.fit(local_centers,sample_weight=np.asarray(local_weights))\n   centers=server.cluster_centers_\n  prototypes.append(np.asarray(centers,dtype=np.float32))\n return prototypes\n\ndef compute_scores(base,z,protos,train_pos,mode):\n if mode==\'none\':return base,None\n bp,ap=protos\n d0=np.min(((z[:,None,:]-bp[None,:,:])**2).sum(-1),axis=1)\n d1=np.min(((z[:,None,:]-ap[None,:,:])**2).sum(-1),axis=1)\n # Training-only standardization: benign proximity vs known attack proximity.\n train_d0=d0[train_pos];scale=max(float(np.quantile(train_d0,.9)-np.quantile(train_d0,.1)),1e-5)\n margin=(d0-d1)/scale\n score=CFG[\'score_weight\']*margin+(1-CFG[\'score_weight\'])*np.log(np.clip(base,1e-6,1-1e-6)/np.clip(1-base,1e-6,1))\n return score,dict(prototype_benign=len(bp),prototype_attack=len(ap),distance_scale=scale)\n\ndef metrics(scores,y,ids,thr):\n z=scores[ids]>=thr; yy=y[ids];tn,fp,fn,tp=confusion_matrix(yy,z,labels=[0,1]).ravel()\n return dict(recall=tp/max(1,tp+fn),fpr=fp/max(1,tn+fp),precision=tp/max(1,tp+fp),attack_f1=f1_score(yy,z,zero_division=0),tp=int(tp),fp=int(fp),tn=int(tn),fn=int(fn))\n\ndef run_fold(df,features,held,seed,device):\n seed_all(seed);X,y,family,tr,va,te,clients,manifest=create_fold(df,features,held,seed)\n ix_all=np.r_[tr,va,te]; pos_train=np.arange(len(tr)); records=[];resc=[];raw=[];anchors=[]\n benign_val=va[y[va]==0]\n models={}\n for kind in [\'MLP\',\'KAN\',\'LOCAL\']:\n  t=time.monotonic();model,comm,steps=train_fed(X,y,clients,seed,kind,device)\n  models[kind]=(model,comm,steps,time.monotonic()-t)\n for method in CFG[\'comparison_methods\']:\n  kind=\'MLP\' if method==\'FedAvgMLP\' else \'LOCAL\' if method==\'LocalKAN\' else \'KAN\'\n  model,comm,steps,elapsed=models[kind]\n  if kind==\'LOCAL\':\n   # Simulated local-only: equal-weight average of per-client scores; explicit ensemble comparator.\n   outputs=[infer(m,X,np.arange(len(X)),device)[0] for m in model]\n   base=np.mean(outputs,axis=0);z=infer(model[0],X,np.arange(len(X)),device)[1]\n  else:base,z=infer(model,X,np.arange(len(X)),device)\n  proto=\'multi\' if method==\'FedAvgKAN_MultiProto\' else \'single\' if method==\'FedAvgKAN_SingleProto\' else \'none\'\n  prototypes=fit_proto(z,y,tr,clients,proto,seed) if proto!=\'none\' else None\n  scores,meta=compute_scores(base,z,prototypes,tr,proto)\n  va_neg=scores[benign_val]\n  try:auroc=roc_auc_score(y[te],scores[te]);ap=average_precision_score(y[te],scores[te])\n  except ValueError:auroc=ap=float(\'nan\')\n  for target in CFG[\'thresholds\']:\n   thr=float(np.quantile(va_neg,1-target,method=\'higher\'))\n   m=metrics(scores,y,te,thr)\n   met=dict(seed=seed,heldout_family=held,method=method,val_target_fpr=target,threshold=thr,auroc=auroc,auprc=ap,n_test=len(te),n_unknown=int(y[te].sum()),**m)\n   records.append(met)\n  resc.append(dict(seed=seed,heldout_family=held,method=method,train_steps=steps,communication_bytes=comm + (sum(x.size*4 for x in prototypes) if prototypes else 0),training_seconds=elapsed,parameters=sum(p.numel() for p in (model[0] if isinstance(model,list) else model).parameters()),prototype_type=proto,**(meta or {})))\n  if method in [\'FedAvgMLP\',\'FedAvgKAN\',\'FedAvgKAN_MultiProto\']:\n   # Scores are raw model/prototype scores, NOT test-selected thresholds.\n   rng=np.random.default_rng(seed+15);subset=rng.choice(te,size=min(len(te),2500),replace=False)\n   for i in subset:raw.append(dict(seed=seed,heldout_family=held,method=method,actual_label=int(y[i]),actual_family=str(family[i]),score=float(scores[i])))\n  if method==\'FedAvgKAN_MultiProto\' and prototypes:\n   for cls,centers in enumerate(prototypes):\n    for j,c in enumerate(centers):anchors.append(dict(seed=seed,heldout_family=held,class_label=cls,prototype_id=j,vector=json.dumps(c.tolist())))\n return records,resc,raw,anchors,manifest\n\ndef summarize(df,keys,metric_cols):\n data=df.groupby(keys,dropna=False)[metric_cols].agg([\'mean\',\'std\']).reset_index()\n data.columns=[\'_\'.join(c).strip(\'_\') for c in data.columns]\n return data\n\ndef main():\n ROOT.mkdir(parents=True,exist_ok=True);path,origin=get_dataset_file();df,features,prov=read_dataset(path)\n families={str(x) for x in df[\'family\'].unique() if str(x).lower() not in [\'normal\',\'benign\']}\n selected=[x for x in CFG[\'families\'] if x in families and int((df[\'family\']==x).sum())>=20]\n if not selected:\n  # Support spelling variants, but report actual picked families, never silently pretend targets exist.\n  priorities=[\'port\',\'ddos\',\'sql\',\'ransom\',\'mitm\']\n  for q in priorities:\n   selected += [f for f in sorted(families) if q in f.lower() and f not in selected]\n  selected=selected[:4]\n if len(selected)<2:raise ValueError(\'Need >=2 held-out attack types with sufficient support; available: \'+repr(df[\'family\'].value_counts().to_dict()))\n print(\'Dataset:\',path,\'rows=\',len(df),\'families=\',selected,\'features=\',len(features),flush=True)\n jsave(OUT/\'config.json\',CFG);jsave(OUT/\'dataset_provenance.json\',dict(**prov,source=str(path),origin=origin,selected_folds=selected))\n all_metrics=[];all_resources=[];all_raw=[];all_proto=[];manifests=[]\n device=\'cuda:0\' if torch.cuda.is_available() else \'cpu\'\n for fold in selected:\n  for seed in CFG[\'seeds\']:\n   print(\'RUN\',fold,\'seed\',seed,device,flush=True)\n   m,r,p,pr,manifest=run_fold(df,features,fold,seed,device)\n   all_metrics.extend(m);all_resources.extend(r);all_raw.extend(p);all_proto.extend(pr);manifests.append(manifest)\n   pd.DataFrame(all_metrics).to_csv(OUT/\'all_metrics.csv\',index=False)\n   pd.DataFrame(all_resources).to_csv(OUT/\'all_resources.csv\',index=False)\n jsave(OUT/\'fold_manifest.json\',manifests)\n md=pd.DataFrame(all_metrics);rd=pd.DataFrame(all_resources)\n summary=summarize(md,[\'heldout_family\',\'method\',\'val_target_fpr\'],[\'recall\',\'fpr\',\'precision\',\'attack_f1\',\'auroc\',\'auprc\']);summary.to_csv(OUT/\'summary_mean_std.csv\',index=False)\n pd.DataFrame(all_raw).to_csv(OUT/\'prediction_samples.csv\',index=False)\n pd.DataFrame(all_proto).to_csv(OUT/\'prototype_vectors.csv\',index=False)\n pd.DataFrame(Counter({\'total_runs\':len(md),\'folds\':len(selected),\'seeds\':len(CFG[\'seeds\'])}).items(),columns=[\'item\',\'count\']).to_csv(OUT/\'run_counts.csv\',index=False)\n fig,axes=plt.subplots(1,2,figsize=(12,4))\n for j,(metric,title) in enumerate([(\'recall_mean\',\'Unknown attack recall\'),(\'fpr_mean\',\'Observed test FPR\')]):\n  sub=summary[summary.val_target_fpr==.05]\n  for method,g in sub.groupby(\'method\'):\n   axes[j].plot(g.heldout_family,g[metric],marker=\'o\',label=method)\n  axes[j].set_title(title+\' (Val FPR target 5%)\');axes[j].tick_params(axis=\'x\',rotation=30);axes[j].set_ylim(0,1)\n axes[0].legend(fontsize=7);fig.tight_layout();fig.savefig(OUT/\'performance_summary.png\',dpi=170);plt.close(fig)\n jsave(OUT/\'run_status.json\',dict(status=\'complete\',n_seed_fold_model=len(rd),n_operating_point_rows=len(md),device=device,data_origin=origin))\n archive=ROOT/CFG[\'archive\']\n with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED,compresslevel=6) as z:\n  for p in sorted(OUT.rglob(\'*\')):\n   if p.is_file():z.write(p,arcname=p.relative_to(ROOT))\n print(\'COMPLETE\',archive,archive.stat().st_size,flush=True)\n\nif __name__==\'__main__\':\n try:main()\n except Exception:\n  traceback.print_exc();sys.exit(1)\n'
(work / "edge_iiot_engine.py").write_text(engine_source, encoding="utf8")
print("Wrote engine:", work / "edge_iiot_engine.py")

In [ ]:
from pathlib import Path
launcher_source = '"""Run the self-contained Edge-IIoTset benchmark using both available Kaggle T4 GPUs."""\nimport os, sys, subprocess, json, zipfile, time\nfrom pathlib import Path\nimport pandas as pd\nimport numpy as np\nimport torch\nROOT=Path(\'/kaggle/working\') if Path(\'/kaggle/working\').exists() else Path(os.getenv(\'EDGE_BENCH_ROOT\',\'/mnt/data/edge_benchmark_launch\'))\nSCRIPT=ROOT/\'edge_iiot_engine.py\'\nif not SCRIPT.exists():\n SCRIPT=Path(\'/mnt/data/EdgeIIoT_OpenSet_Benchmark_Engine.py\')\nif not SCRIPT.exists():raise FileNotFoundError(\'Embedded engine missing.\')\nsmoke=os.getenv(\'EDGE_BENCH_SMOKE\')==\'1\'\nn_gpu=torch.cuda.device_count()\nif n_gpu>=2 and not smoke:\n groups=[\'0,42\',\'3407\'];devices=[\'0\',\'1\']\nelse:\n groups=[\'0\' if smoke else \'0,42,3407\'];devices=[\'0\' if n_gpu else \'\']\nworkers=[]\nprint(f\'Available CUDA GPUs: {n_gpu}; launching {len(groups)} worker(s).\',flush=True)\nfor i,(seeds,dev) in enumerate(zip(groups,devices)):\n dest=ROOT/\'edge_iiot_workers\'/f\'worker_{i}\'\n dest.mkdir(parents=True,exist_ok=True)\n env=os.environ.copy();env[\'EDGE_BENCH_ROOT\']=str(dest)\n if not smoke:env[\'EDGE_BENCH_SEEDS\']=seeds\n if dev:env[\'CUDA_VISIBLE_DEVICES\']=dev\n log=open(dest/\'run.log\',\'w\')\n p=subprocess.Popen([sys.executable,str(SCRIPT)],stdout=log,stderr=subprocess.STDOUT,env=env)\n workers.append((p,log,dest,seeds))\nfailed=[]\nfor p,log,dest,seeds in workers:\n rc=p.wait();log.close();print(\'Worker\',seeds,\'exit\',rc,flush=True)\n if rc:\n  failed.append((seeds,rc,(dest/\'run.log\').read_text()[-3000:]))\nif failed:raise RuntimeError(\'At least one worker failed: \'+str(failed))\nout=ROOT/\'EdgeIIoT_OpenSet_Results\';out.mkdir(parents=True,exist_ok=True)\nfor name in [\'all_metrics.csv\',\'all_resources.csv\',\'prediction_samples.csv\',\'prototype_vectors.csv\']:\n parts=[]\n for p,l,d,s in workers:\n  f=d/\'EdgeIIoT_OpenSet_Results\'/name\n  if f.exists() and f.stat().st_size:parts.append(pd.read_csv(f))\n if parts:pd.concat(parts,ignore_index=True).to_csv(out/name,index=False)\nmet=pd.read_csv(out/\'all_metrics.csv\');res=pd.read_csv(out/\'all_resources.csv\')\nassert set(met.seed.unique())==({0} if smoke else {0,42,3407}), \'Seed coverage mismatch\'\nmetrics=[\'recall\',\'fpr\',\'precision\',\'attack_f1\',\'auroc\',\'auprc\']\nsumdf=met.groupby([\'heldout_family\',\'method\',\'val_target_fpr\'])[metrics].agg([\'mean\',\'std\']).reset_index()\nsumdf.columns=[\'_\'.join(x).strip(\'_\') for x in sumdf.columns]\nsumdf.to_csv(out/\'summary_mean_std.csv\',index=False)\nfor name in [\'config.json\',\'dataset_provenance.json\',\'performance_summary.png\']:\n f=workers[0][2]/\'EdgeIIoT_OpenSet_Results\'/name\n if f.exists():\n  import shutil\n  shutil.copy2(f,out/name)\nmanifest=[]\nfor p,l,d,s in workers:\n f=d/\'EdgeIIoT_OpenSet_Results\'/\'fold_manifest.json\'\n if f.exists():manifest.extend(json.loads(f.read_text()))\n(out/\'fold_manifest.json\').write_text(json.dumps(manifest,indent=2,default=str))\n(out/\'run_status.json\').write_text(json.dumps(dict(status=\'complete\',gpu_count=n_gpu,workers=len(workers),seeds=sorted(met.seed.unique().tolist()),distinct_folds=sorted(met.heldout_family.unique().tolist()),distinct_methods=sorted(met.method.unique().tolist()),evaluation_rows=len(met),training_rows=len(res)),indent=2))\nfor i,(p,l,d,s) in enumerate(workers):\n import shutil\n shutil.copy2(d/\'run.log\',out/f\'worker_{i}.log\')\narchive=ROOT/\'EdgeIIoT_OpenSet_Results.zip\'\nwith zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED,compresslevel=6) as z:\n for f in sorted(out.rglob(\'*\')):\n  if f.is_file():z.write(f,f.relative_to(ROOT))\nprint(\'RESULT_ARCHIVE:\',archive,\'size:\',archive.stat().st_size,flush=True)\nprint(\'RUN_STATUS:\',(out/\'run_status.json\').read_text(),flush=True)\n'
(work / "edge_iiot_launcher.py").write_text(launcher_source, encoding="utf8")
print("Wrote launcher:", work / "edge_iiot_launcher.py")

In [ ]:
import subprocess, sys, os
proc = subprocess.run([sys.executable,str(work/'edge_iiot_launcher.py')],cwd=str(work),env=os.environ.copy(),check=False)
if proc.returncode != 0:raise RuntimeError(f'Benchmark failed (exit {proc.returncode}); inspect logs and Kaggle output.')
assert (work/'EdgeIIoT_OpenSet_Results.zip').is_file()
print('ZIP READY:',work/'EdgeIIoT_OpenSet_Results.zip')

In [ ]:
import pandas as pd
p=work/'EdgeIIoT_OpenSet_Results'/'summary_mean_std.csv'
summary=pd.read_csv(p)
display(summary[['heldout_family','method','val_target_fpr','recall_mean','fpr_mean','auroc_mean','auprc_mean']].round(4))